In [ ]:
%%time
import sys
print(sys.version)
print(sys.executable)
%pip install rasterio numpy pandas geopandas scipy dask[distributed] bokeh rioxarray xarray pathlib openpyxl
%pip show rasterio numpy pandas geopandas scipy dask bokeh rioxarray xarray pathlib openpyxl

In [ ]:
%%time
import pandas as pd
import os
from collections import defaultdict, Counter
import datetime
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import rasterio.mask
import shutil  # This module allows for file copying
from rasterio.warp import reproject, Resampling
from pathlib import Path

# Set the base directory 
base_dir = Path("F:/Master_Thesis_EQ62")  # Adjusted the Base Directory
num_workers = 12  # ปรับจำนวน Thread ตามความแรงของ CPU และความเร็ว Disk
threads_per_worker = 2
memory_limit = '5GB'

# Define paths for input and output directories
EPSG_target = 32647

In [ ]:
%%time
from pathlib import Path
#Range of date interesting for the analysis
date_start = datetime.date(2019, 10, 6)
date_end = datetime.date(2020, 1, 4)
threshold_limit = 0
# Input and Output paths for Pre image processing
Input_dir_Hotspot = base_dir / "Datasource/EQ/Hotspot/input/fire_archive_M-C61_635379.csv"
Output_dir_Hotspot = base_dir / "Datasource/EQ/Hotspot/output/Hotspot"
Output_dir_Hotspotcluster = base_dir / "Datasource/EQ/Hotspot/output/HotspotCluster"
output_folder_SHP = base_dir / "Datasource/EQ/Hotspot/output/HotspotCluster_SHP"
reference_raster_path = base_dir / "Datasource/EQ/Hotspot/input/MOD11A1_LST_Day_1km_2009_273.tif"
PTR_output = base_dir / "Datasource/EQ/Hotspot/output/PointtoRaster"
FB_output = base_dir / "Datasource/EQ/Hotspot/output/MaskingLayer"
reportpath = base_dir / "Report"

#Input and Output paths for Thermal Anomaly
Input_Raw_MODIS = base_dir / "Datasource/MODIStsp/MODIStsp_Celsius"
TempwithoutError = base_dir / "Datasource/MODIStsp/TempwithoutError"
DeltaT_LC = base_dir / "Datasource/MODIStsp/DeltaTempLC"
RETIRA_LC = base_dir / "Datasource/TA/OriginalTA_LC"
RETIRA_Majority_LC = base_dir / "Datasource/TA/OriginalTA_Majority_LC"
RETIRA_Continue_LC = base_dir / "Datasource/TA/OriginalTA_Continue_LC"
LC_folder= base_dir / "Datasource/MODIStsp/LU/LandCover_Type_Yearly_500m_v61/LC1"
outputLC_Folder = base_dir / "Datasource/Landcover/LC"


#AOI Clip with Shapefile
AOI_shp = base_dir / "Datasource/Master_Thesis_AOI/AOI_Thesis.shp"
RETIRA_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_LC_Clip_Final"
RETIRA_Majority_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_Majority_LC_Clip_Final"
RETIRA_Continue_LC_Clip_Final = base_dir / "Datasource/TA/OriginalTA_Continue_LC_Clip_Final"

In [ ]:
%%time
input_OD = base_dir / "Datasource/MODIStsp/MOD/Surf_Temp_Daily_1Km_v61/LST_Day_1km"
input_ON = base_dir / "Datasource/MODIStsp/MOD/Surf_Temp_Daily_1Km_v61/LST_Night_1km"
input_YD = base_dir / "Datasource/MODIStsp/MYD/Surf_Temp_Daily_1Km_v61/LST_Day_1km"
input_YN = base_dir / "Datasource/MODIStsp/MYD/Surf_Temp_Daily_1Km_v61/LST_Night_1km"

K TO C

In [ ]:
%%time
import os
import numpy as np
import rasterio
import geopandas as gpd  # <--- ต้องมีตัวนี้เพื่ออ่านไฟล์ .shp
from rasterio.mask import mask
import gc
from dask.distributed import Client, LocalCluster
from pathlib import Path
from shapely.geometry import mapping # <--- สำคัญมาก ต้องใช้แปลง format

# ───── [1] DASK SETUP ───────────────────────────────────────────────
try:
    client.close()
    cluster.close()
except:
    pass

cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
client = Client(cluster)

# ───── [2] AOI PREPARATION (อ่านไฟล์ SHP) ──────────────────────────
gdf_aoi = None
aoi_shapes = None # ตัวแปรนี้แหละที่จะส่งให้ Worker

if 'AOI_shp' in locals() or 'AOI_shp' in globals():
    aoi_path = Path(AOI_shp)
    ref_path = Path(reference_raster_path)
    
    if aoi_path.exists() and ref_path.exists():
        try:
            # 1. อ่านไฟล์ Shapefile
            gdf_aoi = gpd.read_file(aoi_path)
            
            # 2. อ่าน CRS จาก Reference Raster (LST 1km)
            with rasterio.open(ref_path) as ref_src:
                ref_crs = ref_src.crs
            
            # 3. ตรวจสอบและแปลงระบบพิกัด (Reproject)
            if gdf_aoi.crs != ref_crs:
                print(f"🔄 CRS Mismatch: แปลง AOI จาก {gdf_aoi.crs} ⮕ {ref_crs}")
                gdf_aoi = gdf_aoi.to_crs(ref_crs)
            else:
                print(f"✅ CRS Match: AOI และ Reference ตรงกัน ({ref_crs})")
            
            # 4. แปลงจาก GeoDataFrame เป็น List of Geometries (GeoJSON format)
            # แก้ปัญหา 'str' object has no attribute 'get' และช่วยให้ส่งผ่าน Dask ได้ง่าย
            aoi_shapes = [mapping(g) for g in gdf_aoi.geometry]
            
            print(f"📌 AOI Ready: {aoi_path.name} ({len(aoi_shapes)} shapes)")
            
        except Exception as e:
            print(f"⚠️ เกิดข้อผิดพลาดในการเตรียม AOI: {e}")
            gdf_aoi = None
            aoi_shapes = None
    else:
        # กรณีหาไฟล์ไม่เจอ ให้แจ้งพาธที่ระบบพยายามหา
        if not aoi_path.exists(): print(f"ℹ️ ไม่พบไฟล์ AOI ที่: {aoi_path}")
        if not ref_path.exists(): print(f"ℹ️ ไม่พบไฟล์ Reference ที่: {ref_path}")
else:
    print("ℹ️ ไม่มีการประกาศตัวแปร AOI_shp ไว้ในระบบ")

# ───── [3] WORKER FUNCTION ──────────────────────────────────────────

def process_kelvin_to_celsius_worker(file_info, output_folder, shapes=None):
    input_path, file_name = file_info
    out_path = os.path.join(output_folder, file_name)
    
    try:
        with rasterio.open(input_path) as src:
            # ตรวจสอบ CRS (ถ้ามี shapes ต้องเช็คว่าตรงกับภาพไหม)
            # ในที่นี้สมมติว่า CRS ตรงกัน หรือผู้ใช้จัดการมาแล้ว
            
            if shapes is not None:
                # 1. Clip ภาพ
                out_image, out_transform = mask(src, shapes, crop=True)
                data = out_image[0].astype(np.float32)
                meta = src.meta.copy()
                meta.update({
                    "height": data.shape[0],
                    "width": data.shape[1],
                    "transform": out_transform
                })
            else:
                # 1. อ่านปกติ (ไม่ตัด)
                data = src.read(1).astype(np.float32)
                meta = src.meta.copy()

            # 2. แปลง Kelvin เป็น Celsius
            nodata = src.nodata
            valid_mask = (data != nodata)
            data[valid_mask] = data[valid_mask] - 273.15
            
            # 3. อัปเดต Meta สำหรับเขียนไฟล์
            meta.update({
                'dtype': 'float32',
                'compress': 'lzw'
            })

            with rasterio.open(out_path, 'w', **meta) as dst:
                dst.write(data, 1)
                
        mode = "Clipped" if shapes else "Full-Extent"
        return f"✔ {mode}: {file_name}"
    
    except Exception as e:
        return f"❌ Error {file_name}: {str(e)}"

# ───── [4] MAIN EXECUTION ───────────────────────────────────────────
input_output_pairs = [
    (input_OD, Input_Raw_MODIS),
    (input_ON, Input_Raw_MODIS),
    (input_YD, Input_Raw_MODIS),
    (input_YN, Input_Raw_MODIS),
]

print(f"🚀 เริ่มประมวลผลด้วย {num_workers} Workers...")

for idx, (input_dir, output_dir) in enumerate(input_output_pairs, 1):
    if not os.path.exists(input_dir):
        continue
        
    print(f"\n--- ชุดที่ {idx}: {os.path.basename(input_dir)} ---")
    os.makedirs(output_dir, exist_ok=True)
    
    tif_files = [(os.path.join(input_dir, f), f) for f in os.listdir(input_dir) if f.lower().endswith('.tif')]
    
    if not tif_files:
        continue
    
    # ส่งงานให้ Dask
    futures = client.map(process_kelvin_to_celsius_worker, tif_files, 
                         output_folder=output_dir, 
                         shapes=aoi_shapes)
    
    results = client.gather(futures)
    gc.collect()
    print(f"✨ เสร็จสิ้น {len(results)} ไฟล์")

print(f"\n✅ ประมวลผลทั้งหมดเรียบร้อย!")

# ปิด Dask
client.close()
cluster.close()

EXCEL TO Excel Per DATE

In [ ]:
%%time
import os
import pandas as pd
from dask.distributed import Client, LocalCluster
from pathlib import Path

# === 1. Setup Dask Cluster (ตัวแทน ThreadPoolExecutor) ===
if __name__ == "__main__":
    # ตั้งสเปกเหมือนเดิม แต่คุม RAM ได้ด้วย
    cluster = LocalCluster(
        n_workers=num_workers,           # เทียบเท่า max_workers=8
        threads_per_worker=threads_per_worker,  # 1 thread ต่อ 1 process (เลี่ยง GIL ได้ดีกว่า)
        memory_limit=memory_limit     # จำกัด RAM ต่อหัว ไม่ให้รวมกันแล้วเกินเครื่อง
    )
    client = Client(cluster)
    print(f"🚀 Dashboard: {client.dashboard_link}")

    # === 2. Config Paths ===
    input_file = Input_dir_Hotspot
    output_dir = Output_dir_Hotspotcluster
    os.makedirs(output_dir, exist_ok=True)

    # === 3. Worker Function (Pandas ล้วนๆ) ===
    def pandas_worker(df_group, group_name, out_path):
        """รับ Pandas DataFrame มาเขียน Excel"""
        try:
            date_obj, sat, dn = group_name
            filename = f"{date_obj.strftime('%d%m%Y')}_{sat}_{dn}.xlsx"
            file_path = os.path.join(out_path, filename)
            
            # ใช้ Pandas write ปกติ
            df_group.to_excel(file_path, index=False, engine='openpyxl')
            return f"Success: {filename}"
        except Exception as e:
            return f"Error: {str(e)}"

    # === 4. Main Process ===
    print("📑 Loading data with Pandas...")
    df = pd.read_csv(input_file)
    df['acq_date'] = pd.to_datetime(df['acq_date'], dayfirst=True)

    print("📦 Grouping...")
    grouped = df.groupby(['acq_date', 'satellite', 'daynight'])

    print("🔥 Submitting tasks to Dask Distributed...")
    futures = []
    for name, group in grouped:
        # ใช้ client.submit แทน executor.submit
        # Dask จะส่ง Pandas group นี้ไปรันที่ Worker ตัวที่ว่างอยู่
        future = client.submit(pandas_worker, group, name, str(output_dir))
        futures.append(future)

    # === 5. Gather Results ===
    print(f"⏳ Waiting for {len(futures)} tasks to finish...")
    results = client.gather(futures) # คล้ายๆ list(executor.map)

    print(f"✅ Done! Created {len(results)} files.")
    
    client.close()
    cluster.close()

Excel to SHP

In [ ]:
%%time
import os
import datetime
import pandas as pd
import geopandas as gpd
from collections import Counter
import dask
from dask.distributed import Client, LocalCluster

# === 1. เตรียม Path (ใช้ตัวแปรเดิมของพี่) ===
os.makedirs(output_folder_SHP, exist_ok=True)
os.makedirs(reportpath, exist_ok=True)
# === 2. สร้าง Worker Function (ยก Logic พี่มาวางเป๊ะๆ) ===
@dask.delayed
def original_process_logic(excel_file, target_epsg, out_folder):
    try:
        # --- [START: Logic เดิมของพี่] ---
        df = pd.read_excel(excel_file)
        if df.empty:
            return None

        if 'acq_date' in df.columns:
            df = df.drop(columns=['acq_date'])

        # สร้าง GeoDataFrame และแปลง CRS
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
        gdf.set_crs(epsg=4326, inplace=True)
        gdf = gdf.to_crs(epsg=target_epsg)

        # Logic การแกะชื่อไฟล์และวันที่ (ยกมาเป๊ะๆ)
        date_str = os.path.basename(excel_file)[:8]
        date_obj = datetime.datetime.strptime(date_str, '%d%m%Y')
        doy = date_obj.timetuple().tm_yday
        year = date_obj.year
        doy_str = f"{year}_{doy:03d}"

        if "TERRA_D" in excel_file.upper():
            base_name = f"MOD11A1_LST_Day_1km_{doy_str}"
        elif "TERRA_N" in excel_file.upper():
            base_name = f"MOD11A1_LST_Night_1km_{doy_str}"
        elif "AQUA_D" in excel_file.upper():
            base_name = f"MYD11A1_LST_Day_1km_{doy_str}"
        elif "AQUA_N" in excel_file.upper():
            base_name = f"MYD11A1_LST_Night_1km_{doy_str}"
        else:
            base_name = f"UNK_LST_UNK_1km_{doy_str}"

        # บันทึกไฟล์ลง Disk
        output_filename = os.path.join(out_folder, f"{base_name}.shp")
        gdf.to_file(output_filename)
        # --- [END: Logic เดิมของพี่] ---

        # ส่งค่ากลับมาทำ Report
        return {
            'Excel_File': os.path.basename(excel_file),
            'Shapefile': f'{base_name}.shp',
            'satellite': df['satellite'].iloc[0] if 'satellite' in df.columns else "Unknown",
            'daynight': df['daynight'].iloc[0] if 'daynight' in df.columns else "Unknown"
        }
    except Exception as e:
        print(f"❌ Error processing {excel_file}: {e}")
        return None

# === 3. ส่วนการรัน (Main) ===
if __name__ == "__main__":
    # ตั้งค่า Cluster 8 หัวตามสูตรพี่
    cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
    client = Client(cluster)
    print(f"📈 Dashboard: {client.dashboard_link}")

    # ดึงรายชื่อไฟล์
    excel_files = [os.path.join(Output_dir_Hotspotcluster, f) 
                   for f in os.listdir(Output_dir_Hotspotcluster) 
                   if f.endswith('.xlsx')]

    print(f"⚙️ Step: Processing {len(excel_files)} files with Dask...")

    # สร้างรายการงาน (Lazy Tasks)
    tasks = [original_process_logic(f, EPSG_target, output_folder_SHP) for f in excel_files]

    # สั่งรันรวดเดียว
    results = dask.compute(*tasks)

    # === 4. สรุปผล (เอาผลลัพธ์มาปั่น Report) ===
    file_details = []
    satellite_counts = Counter()

    for r in results:
        if r is not None:
            file_details.append({
                'Excel_File': r['Excel_File'],
                'Shapefile': r['Shapefile']
            })
            satellite_counts[f"{r['satellite']} ({r['daynight']})"] += 1

    print(f"\n🏁 Finished! Total files processed: {len(file_details)}")
    for key, count in satellite_counts.items():
        print(f"📊 {key}: {count} files")

    # บันทึก Report (Logic เดิม)
    if file_details:
        details_df = pd.DataFrame(file_details)
        details_report_path = os.path.join(reportpath, 'HotspotCluster_SHP_Report.xlsx')
        details_df.to_excel(details_report_path, index=False)
        print(f"📄 Report saved at: {details_report_path}")

    client.close()
    cluster.close()

SHP TO TIFF BIT MASK

In [ ]:
%%time
import os
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
import dask
from dask.distributed import Client, LocalCluster

# ========== 1. CONFIG & PREPARE ==========
nodata_value = -9999
os.makedirs(PTR_output, exist_ok=True)
os.makedirs(FB_output, exist_ok=True)

if __name__ == "__main__":
    # เริ่มโรงงาน 8 หัวตามสูตรเดิมของมึง
    cluster = LocalCluster(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)
    client = Client(cluster)
    print(f"📈 Dashboard: {client.dashboard_link}")

    # โหลดค่าอ้างอิงครั้งเดียวที่เครื่องแม่
    with rasterio.open(reference_raster_path) as ref_raster:
        ref_data = {
            'transform': ref_raster.transform,
            'crs': ref_raster.crs,
            'width': ref_raster.width,
            'height': ref_raster.height,
            'profile': ref_raster.profile.copy()
        }

    # ⚡ [CACHE] กระจายค่าอ้างอิงไปรอที่ Worker ทุกตัว (ไม่ต้องส่งไปส่งมาใน Graph)
    ref_future = client.scatter(ref_data, broadcast=True)

    # รายชื่อไฟล์ Shapefile
    shapefile_paths = [
        os.path.join(output_folder_SHP, f)
        for f in os.listdir(output_folder_SHP)
        if f.endswith('.shp')
    ]

    # --- Worker Function (ห้ามเปลี่ยนวิธีการคำนวณ!) ---
    @dask.delayed
    def process_raster_dask(shp_path, ref):
        try:
            base_name = os.path.splitext(os.path.basename(shp_path))[0]
            
            # 1. โหลด GDF (ทำที่ Worker เพื่อประหยัดแรมเครื่องแม่)
            gdf = gpd.read_file(shp_path)
            if gdf.empty:
                return {"Shapefile Name": base_name, "Status": "Empty"}

            # 2. [Logic เดิม] Rasterize ใน RAM
            ptr_array = rasterize(
                [(geom, 1) for geom in gdf.geometry],
                out_shape=(ref['height'], ref['width']),
                transform=ref['transform'],
                fill=nodata_value,
                dtype='float32'
            )

            # 3. [Logic เดิม] Apply Masking ทันที
            fb_array = np.where(
                np.isnan(ptr_array) | (ptr_array == nodata_value),
                1, nodata_value
            ).astype('float32')

            # 4. [Logic เดิม] เตรียม Profile
            final_profile = ref['profile'].copy()
            final_profile.update({
                'driver': 'GTiff',
                'height': ref['height'],
                'width': ref['width'],
                'count': 1,
                'dtype': 'float32',
                'crs': ref['crs'],
                'transform': ref['transform'],
                'nodata': nodata_value,
                'compress': 'lzw'
            })

            # 5. [Logic เดิม] เขียนไฟล์ลง Disk
            ptr_path = os.path.join(PTR_output, f"{base_name}_PTR.tif")
            fb_path = os.path.join(FB_output, f"{base_name}_FB.tif")

            with rasterio.open(ptr_path, 'w', **final_profile) as dst:
                dst.write(ptr_array, 1)
            
            with rasterio.open(fb_path, 'w', **final_profile) as dst:
                dst.write(fb_array, 1)

            return {
                "Shapefile Name": base_name,
                "Status": "Success",
                "Pixels_1": int(np.sum(fb_array == 1))
            }

        except Exception as e:
            return {"Shapefile Name": os.path.basename(shp_path), "Status": f"Error: {e}"}

    # สร้าง Tasks
    print(f"📥 เตรียมประมวลผล {len(shapefile_paths)} ไฟล์...")
    tasks = [process_raster_dask(p, ref_future) for p in shapefile_paths]

    # สั่งลุยรวดเดียว
    print("🔥 Dask is computing... รุมสับ Raster ทั่วไทย!")
    results = dask.compute(*tasks)

    # ========== 3. SAVE REPORT ==========
    report_df = pd.DataFrame([r for r in results if r is not None])
    report_df.to_excel(os.path.join(reportpath, 'PTR_Dask_Report.xlsx'), index=False)
    
    client.close()
    cluster.close()
    print("🚀 All processes completed. RAM Cleared!")

Temp Correction without Hotspot

In [ ]:
%%time
import os
import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from concurrent.futures import ThreadPoolExecutor, as_completed

# === 1. CONFIG & PREPARE ===
os.makedirs(TempwithoutError, exist_ok=True)
os.makedirs(reportpath, exist_ok=True)

# Function สำหรับ Reproject โดยรับข้อมูลจาก RAM (Numpy Array)
def reproject_memory_mask(src_array, src_profile, match_src):
    dst_transform = match_src.transform
    dst_crs = match_src.crs
    dst_width = match_src.width
    dst_height = match_src.height
    dst_array = np.empty((dst_height, dst_width), dtype=np.float32)

    reproject(
        source=src_array,
        destination=dst_array,
        src_transform=src_profile['transform'],
        src_crs=src_profile['crs'],
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )
    return dst_array

# === PHASE 1: LOAD ALL MASKS INTO RAM ===
mask_cache = {}
print(f"🚀 Phase 1: Loading Masks (_FB.tif) into RAM...")

if os.path.exists(FB_output):
    mask_files = [f for f in os.listdir(FB_output) if f.endswith('_FB.tif')]
    for mf in mask_files:
        mask_path = os.path.join(FB_output, mf)
        with rasterio.open(mask_path) as m_src:
            # เก็บเฉพาะข้อมูลที่จำเป็นเพื่อประหยัด Memory
            mask_cache[mf] = {
                'array': m_src.read(1),
                'profile': {
                    'transform': m_src.transform,
                    'crs': m_src.crs,
                    'height': m_src.height,
                    'width': m_src.width,
                    'nodata': m_src.nodata
                }
            }
    print(f"✔ Cached {len(mask_cache)} masks in RAM.")
else:
    print("⚠️ FB_output folder not found.")

# === PHASE 2: PROCESSING FUNCTION ===
def process_file_with_cache(input_file):
    try:
        input_path = os.path.join(Input_Raw_MODIS, input_file)
        output_path = os.path.join(TempwithoutError, input_file)
        mask_key = input_file.replace('.tif', '_FB.tif')

        with rasterio.open(input_path) as src:
            input_data = src.read(1)
            meta = src.meta.copy()
            input_nodata = src.nodata if src.nodata is not None else -9999
            meta.update(nodata=input_nodata)

            # ตรวจสอบว่ามี Mask ใน RAM หรือไม่
            if mask_key not in mask_cache:
                shutil.copy(input_path, output_path)
                return {"File Name": input_file, "Status": "Mask Missing (Copied)"}

            # ดึงข้อมูล Mask จาก Cache
            cached_mask = mask_cache[mask_key]
            m_array = cached_mask['array']
            m_prof = cached_mask['profile']
            m_nodata = m_prof['nodata']

            # ตรวจสอบว่าต้อง Reproject หรือไม่
            if (src.transform != m_prof['transform'] or 
                src.crs != m_prof['crs'] or 
                src.shape != (m_prof['height'], m_prof['width'])):
                
                mask_array_final = reproject_memory_mask(m_array, m_prof, src)
            else:
                mask_array_final = m_array

            # Apply masking logic
            # พื้นที่ที่ต้องการ (Valid) คือพื้นที่ที่ mask ไม่ใช่ nodata
            mask_condition = (mask_array_final != m_nodata)
            masked = np.where(mask_condition, input_data, input_nodata)

            with rasterio.open(output_path, "w", **meta) as dest:
                dest.write(masked, 1)

            return {"File Name": input_file, "Status": "Processed"}

    except Exception as e:
        return {"File Name": input_file, "Status": "Error", "Notes": str(e)}

# === PHASE 3: RUN PARALLEL ===
if __name__ == "__main__":
    tif_files = [f for f in os.listdir(Input_Raw_MODIS) if f.endswith('.tif')]
    report_data = []

    print(f"\n🚀 Phase 2: Processing {len(tif_files)} files in parallel...")
    
    # ใช้ ThreadPoolExecutor ตามโครงสร้างที่คุณต้องการ
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(process_file_with_cache, f) for f in tif_files]
        for future in as_completed(futures):
            result = future.result()
            report_data.append(result)
            print(f"✔ {result['File Name']}: {result['Status']}")

    # Save report
    df = pd.DataFrame(report_data)
    report_name = "TemperatureWmasking_RAMMC.xlsx"
    df.to_excel(os.path.join(reportpath, report_name), index=False)
    print(f"\n✅ All Complete! Report saved to: {os.path.join(reportpath, report_name)}")

Landcover / Land-Sea Masking 

Landcover

In [ ]:
%%time
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import Affine
from rasterio.mask import mask  # <--- เพิ่มสำหรับ Clipping
from scipy.stats import mode
import os
import glob
import geopandas as gpd
from pathlib import Path

# ───── [1] AOI PREPARATION (อ่านไฟล์ SHP & เช็ค CRS กับ Reference) ─────
gdf_aoi = None
aoi_shapes = None  # ตัวแปรที่จะส่งให้ rasterio

if 'AOI_shp' in locals() or 'AOI_shp' in globals():
    aoi_path = Path(AOI_shp)
    ref_path = Path(reference_raster_path)
    
    if aoi_path.exists() and ref_path.exists():
        try:
            # 1. อ่าน AOI
            gdf_aoi = gpd.read_file(aoi_path)
            
            # 2. อ่าน CRS จาก Reference Raster (LST 1km)
            with rasterio.open(ref_path) as ref_src:
                ref_crs = ref_src.crs
            
            # 3. แปลง CRS ของ AOI ให้ตรงกับ Reference ทันที
            if gdf_aoi.crs != ref_crs:
                print(f"🔄 CRS Mismatch: แปลง AOI จาก {gdf_aoi.crs} ⮕ {ref_crs}")
                gdf_aoi = gdf_aoi.to_crs(ref_crs)
            else:
                print(f"✅ CRS Match: AOI และ Reference มีพิกัดตรงกัน ({ref_crs})")
            
            # 4. แปลง GeoDataFrame เป็น List of Geometries (GeoJSON format) 
            # ขั้นตอนนี้สำคัญมาก เพื่อแก้ Error 'str' object has no attribute 'get'
            aoi_shapes = [mapping(g) for g in gdf_aoi.geometry]
            print(f"📌 AOI Ready: {aoi_path.name} (พร้อมใช้ตัดภาพ)")
            
        except Exception as e:
            print(f"⚠️ เกิดข้อผิดพลาดในการเตรียม AOI: {e}")
            aoi_shapes = None
    else:
        print(f"ℹ️ ไม่พบไฟล์ AOI หรือ Reference Raster ตามพาธที่ระบุ")

# ───── [2] FUNCTION WITH CLIPPING ───────────────────────────────────

def downscale_snap_and_clip_majority(input_tif, reference_tif, output_tif, shapes=None):
    """
    1. Clip ภาพตาม shapes (ถ้ามี)
    2. Downscale 2x2 ด้วยวิธี Majority (Mode)
    3. Snap ให้ตรงกับ Reference Grid
    """
    with rasterio.open(input_tif) as src:
        # --- ขั้นตอนที่ 1: Clipping (ถ้ามี shapes) ---
        if shapes is not None:
            # ตัดภาพตาม AOI
            out_image, out_transform = mask(src, shapes, crop=True)
            data = out_image[0]
            input_crs = src.crs
            input_nodata = src.nodata
            input_dtype = src.dtypes[0]
            input_transform = out_transform
        else:
            # อ่านปกติ
            data = src.read(1)
            input_transform = src.transform
            input_crs = src.crs
            input_nodata = src.nodata
            input_dtype = src.dtypes[0]

    # --- ขั้นตอนที่ 2: Downscale 2x2 (Majority) ---
    rows, cols = data.shape
    rows_trim = rows - rows % 2
    cols_trim = cols - cols % 2
    data_trimmed = data[:rows_trim, :cols_trim]

    reshaped = data_trimmed.reshape(rows_trim // 2, 2, cols_trim // 2, 2)
    blocks = reshaped.swapaxes(1, 2).reshape(-1, 4)
    
    # คำนวณหาฐานนิยม (Majority)
    mode_result = mode(blocks, axis=1, nan_policy='propagate', keepdims=False)
    downscaled_data = mode_result.mode.reshape(rows_trim // 2, cols_trim // 2)

    # ปรับ Transform ใหม่หลัง Downscale (ขยาย Pixel size เป็น 2 เท่า)
    new_transform = input_transform * Affine.scale(2, 2)

    # --- ขั้นตอนที่ 3: Snapping to Reference Grid ---
    with rasterio.open(reference_tif) as ref:
        ref_profile = ref.profile
        ref_crs = ref.crs
        ref_transform = ref.transform
        ref_width = ref.width
        ref_height = ref.height
        ref_nodata = ref.nodata

    snapped_array = np.full((ref_height, ref_width), 
                            ref_nodata if ref_nodata is not None else 0, 
                            dtype=input_dtype)

    reproject(
        source=downscaled_data,
        destination=snapped_array,
        src_transform=new_transform,
        src_crs=input_crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.nearest
    )

    # --- ขั้นตอนที่ 4: Write Output ---
    out_profile = ref_profile.copy()
    out_profile.update({
        'driver': 'GTiff',
        'dtype': input_dtype,
        'nodata': input_nodata if input_nodata is not None else ref_nodata,
        'width': ref_width,
        'height': ref_height,
        'transform': ref_transform,
        'crs': ref_crs,
        'compress': 'lzw'
    })

    with rasterio.open(output_tif, 'w', **out_profile) as dst:
        dst.write(snapped_array.astype(input_dtype), 1)

# ───── [3] MAIN LOOP ────────────────────────────────────────────────

input_folder = LC_folder
output_folder = outputLC_Folder
reference_path = reference_raster_path

os.makedirs(output_folder, exist_ok=True)
input_files = glob.glob(os.path.join(input_folder, "*.tif"))

print(f"🚀 เริ่มประมวลผลทั้งหมด {len(input_files)} ไฟล์...")

for input_tif in input_files:
    file_name = os.path.basename(input_tif)
    output_path = os.path.join(output_folder, file_name)
    
    try:
        # เรียกใช้ฟังก์ชันพร้อมส่งค่า shapes (ถ้าไม่มีจะเป็น None)
        downscale_snap_and_clip_majority(input_tif, reference_path, output_path, shapes=aoi_shapes)
        print(f"✔ Done: {file_name}")
    except Exception as e:
        print(f"❌ Error {file_name}: {e}")

print("\n✅ ทุกขั้นตอนเสร็จสมบูรณ์!")

Delta Temperature 

With LC Grouping

In [ ]:
%%time
import xarray as xr
import rioxarray
import numpy as np
import os
import glob
import re
import gc

def solve_zonal_thermal_robust():
    temp_dir = TempwithoutError
    lclw_dir = outputLC_Folder
    output_dir = DeltaT_LC
    os.makedirs(output_dir, exist_ok=True)

    # 1. ดึงไฟล์ Temp ทั้งหมดมาเป็นตัวตั้งต้น
    t_files = glob.glob(os.path.join(temp_dir, "*.tif"))
    
    print(f"พบไฟล์อุณหภูมิทั้งหมด: {len(t_files)} ไฟล์")

    for t_path in t_files:
        # 2. ดึง "ปี" ออกมาเพื่อไปหาไฟล์ LCLW ที่คู่กัน
        match = re.search(r'\d{4}', os.path.basename(t_path))
        if not match:
            continue
        year = match.group()
        
        # 3. ค้นหาไฟล์ LCLW ที่มีปีเดียวกันในชื่อไฟล์
        # วิธีนี้จะทำให้ไม่หลุดลูปแม้จำนวนไฟล์ในโฟลเดอร์จะไม่เท่ากัน
        l_search = glob.glob(os.path.join(lclw_dir, f"*{year}*.tif"))
        
        if not l_search:
            print(f"⚠️ ข้ามปี {year}: ไม่พบไฟล์ LCLW ที่มีปีตรงกัน")
            continue
            
        l_path = l_search[0] # เลือกไฟล์แรกที่เจอ
        base_name = os.path.basename(t_path).replace(".tif", "")
        diff_out_path = os.path.join(output_dir, f"{base_name}_difT.tif")

        # ปิดเงื่อนไขข้ามไฟล์ (เพื่อให้รันใหม่ทั้งหมด)
        # if os.path.exists(diff_out_path): continue 

        print(f"--- Processing Year {year}: {base_name} :{l_path}---")

        try:
            # --- ส่วนการคำนวณเหมือนเดิม (Manual RAM Clear) ---
            t_ds = rioxarray.open_rasterio(t_path).sel(band=1).drop_vars('band')
            l_ds = rioxarray.open_rasterio(l_path).sel(band=1).drop_vars('band').rio.reproject_match(t_ds)
            
            t_ds = t_ds.where(t_ds > 0)
            means = t_ds.groupby(l_ds).mean()
            
            zonal_mean_raster = xr.full_like(t_ds, np.nan)
            for zone_val in means.coords[means.dims[0]].values:
                val = float(means.sel({means.dims[0]: zone_val}))
                zonal_mean_raster = zonal_mean_raster.where(l_ds != zone_val, val)

            # บันทึกไฟล์
            mean_out_path = os.path.join(output_dir, f"{base_name}_mean.tif")
            zonal_mean_raster.rio.to_raster(mean_out_path, compress='LZW')
            
            diff_raster = t_ds - zonal_mean_raster
            diff_raster.rio.to_raster(diff_out_path, compress='LZW')

            # เคลียร์แรม
            del t_ds, l_ds, means, zonal_mean_raster, diff_raster
            gc.collect()

        except Exception as e:
            print(f"❌ Error at {year}: {e}")

    print("🚀 ประมวลผลเสร็จสิ้น!")

if __name__ == "__main__":
    solve_zonal_thermal_robust()

Original

RETIRA Index

In [ ]:
%%time
%pip install xarray rioxarray numpy dask
%pip install dask[distributed]
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
import dask
from dask.distributed import Client, LocalCluster

# ───── 1. CONFIG (ดึงจากที่คุณกำหนด) ──────────────────────────────────

# สร้างลิสต์ DOY และปี จากช่วงวันที่กำหนด
doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

# Dask Settings
client = Client(n_workers=num_workers, threads_per_worker=threads_per_worker, memory_limit=memory_limit)

scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    # ค้นหาไฟล์ในโฟลเดอร์ DeltaT อ้างอิงตาม Pattern ชื่อไฟล์
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

@dask.delayed
def calculate_retira_index_delayed(baseline_paths, current_path, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999
        # เปิดไฟล์แบบ Lazy loading ด้วย Dask
        baseline_ds = [rioxarray.open_rasterio(p, chunks={'x': 512, 'y': 512}, mask_and_scale=True) for p in baseline_paths]
        stack = xr.concat(baseline_ds, dim="time")
        current_da = rioxarray.open_rasterio(current_path, chunks={'x': 512, 'y': 512}, mask_and_scale=True)
        
        # คำนวณสถิติ
        mean_da = stack.mean(dim="time")
        std_da = stack.std(dim="time", ddof=1)
        count_da = stack.notnull().sum(dim="time")

        # Logic RETIRA
        z_da = (current_da - mean_da) / std_da
        invalid_mask = (std_da == 0) | (current_da.isnull()) | (count_da < 2)
        z_da = z_da.where(~invalid_mask, nodata_val)
        
        # ทำแบบเดียวกันกับ Mean และ STD (เผื่อต้องการให้ ArcGIS โชว์ NoData เหมือนกัน)
        mean_da = mean_da.where(mean_da.notnull(), nodata_val)
        std_da = std_da.where(std_da.notnull(), nodata_val)
        
        mean_da.rio.to_raster(os.path.join(out_dir, f"{prefix}_Mean.tif"))
        std_da.rio.to_raster(os.path.join(out_dir, f"{prefix}_STD.tif"))
        
        
        output_path = os.path.join(out_dir, f"{prefix}_RETIRA.tif")
        
        z_da.rio.write_nodata(nodata_val, encoded=True, inplace=True)
        z_da.rio.to_raster(output_path)
        
        return f"Done: {prefix}"
    except Exception as e:
        return f"Error: {str(e)}"

# ───── 3. MAIN LOOP (แบ่ง Scenario ชัดเจน) ──────────────────────────

tasks = []

# 1. วนลูปตาม Scenario (MOD_Day, MOD_Night, etc.)
for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    # กำหนดโฟลเดอร์ปลายทางแยกตาม Scenario
    current_out_dir = os.path.join(RETIRA_LC, f"{s}_{d}")
    
    # 2. วนลูปตามช่วงวันที่ (DOY) ที่กำหนดใน config
    for year, doy in doy_year_pairs:
        target_file = find_file(DeltaT_LC, s, d, year, doy)
        
        if target_file:
            tag = f"{year}_{doy:03d}"
            
            # 3. วนลูปตามจำนวนปี Baseline (3, 5, 10 ปี)
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                base_years = range(year - (n_years - 1), year + 1)
                valid_paths = []
                
                for by in base_years:
                    p = find_file(DeltaT_LC, s, d, by, doy)
                    if p: valid_paths.append(p)
                
                if len(valid_paths) >= 2:
                    # เพิ่มงานเข้า Dask Queue
                    task = calculate_retira_index_delayed(
                        valid_paths, target_file, current_out_dir, tag, lbl, s, d
                    )
                    tasks.append(task)

print(f"รวบรวมงานสำเร็จ: {len(tasks)} รายการ")
print(f"เริ่มประมวลผลขนานด้วย Dask (Workers: {num_workers})...")

# สั่ง Execute ทุกอย่างพร้อมกัน
results = dask.compute(*tasks)

# แสดงผลลัพธ์ 5 อันดับแรก
for res in results[:5]:
    print(res)

client.close()
print(f"=== เสร็จสิ้นการทำงาน ===")

Majority

LC

In [ ]:
%%time
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
from scipy.stats import mode
import warnings
import numpy as np

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice")

# ───── 1. CONFIGURATION (ตัด Dask ออกแล้ว) ──────────────────────────
scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def find_lc_file(root, year):
    pattern = f"*LC*{year}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def calculate_retira_with_debug(baseline_paths, lc_paths, current_path, current_lc_path, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999.0
        
        # 1. โหลด Reference (ไฟล์ต้นฉบับพันกว่าเมตร)
        ref_da = rioxarray.open_rasterio(current_path, mask_and_scale=True).squeeze()
        if "band" in ref_da.dims: ref_da = ref_da.drop_vars("band")

        # 2. เตรียมข้อมูล (Match พิกัด)
        # เก็บจำนวนปีสูงสุดที่ส่งเข้ามา
        max_years_count = len(baseline_paths) 
        
        baseline_ds = []
        for p in baseline_paths:
            da = rioxarray.open_rasterio(p, mask_and_scale=True).squeeze()
            if da.shape != ref_da.shape:
                da = da.rio.reproject_match(ref_da, resampling=1)
            baseline_ds.append(da)
            
        lc_ds = []
        for p in lc_paths:
            da = rioxarray.open_rasterio(p).squeeze()
            da = da.rio.reproject_match(ref_da, resampling=0)
            lc_ds.append(da)
            
        current_da = ref_da 
        current_lc = rioxarray.open_rasterio(current_lc_path).squeeze().rio.reproject_match(ref_da, resampling=0)

        # 3. Stack & Filter
        stack_dt = xr.concat(baseline_ds, dim="time")
        stack_lc = xr.concat(lc_ds, dim="time")
        
        # หา Majority LC (ตรรกะเดิมที่นายต้องการเช็ค)
        m = mode(stack_lc.values, axis=0, keepdims=False)
        maj_val = getattr(m, 'mode', m[0])
        majority_lc = xr.DataArray(maj_val, coords=current_da.coords, dims=current_da.dims)

        # สร้าง Mask: LC ต้องตรงทั้ง Majority และปีปัจจุบัน
        valid_mask = (stack_lc == majority_lc) & (majority_lc == current_lc)
        filtered_dt = stack_dt.where(valid_mask)

        # 4. คำนวณสถิติ
        # Count = จำนวนปีที่ 'รอด' ผ่านเงื่อนไข LC มาได้
        count_da = filtered_dt.notnull().sum(dim="time")
        # MaxYears = จำนวนปีทั้งหมดที่พยายามจะใช้ (ค่าคงที่ทั้งแผ่น)
        max_years_da = xr.full_like(ref_da, fill_value=max_years_count)
        
        mean_da = filtered_dt.mean(dim="time")
        std_da = filtered_dt.std(dim="time", ddof=1)

        # 5. คำนวณ RETIRA
        z_da = (current_da - mean_da) / std_da
        
        # เงื่อนไขการเซฟ: ถ้าจุดไหนข้อมูลไม่ถึง 2 ปี ให้เป็น NoData ในไฟล์ผลลัพธ์หลัก
        final_mask = (count_da < 2) | (std_da == 0) | (current_da.isnull())
        
        # 6. เตรียม Output
        outputs = {
            "RETIRA": z_da.where(~final_mask, nodata_val),
            "Count": count_da.astype(np.float32),      # จำนวนปีที่ใช้ได้จริง
            "MaxYears": max_years_da.astype(np.float32) # จำนวนปีตั้งต้น (เช่น 3, 5, 10)
        }

        # 7. บันทึกไฟล์
        for name, da in outputs.items():
            res = da.load()
            res.rio.write_nodata(nodata_val, encoded=True, inplace=True)
            res.rio.set_crs(ref_da.rio.crs, inplace=True)
            
            out_file = os.path.join(out_dir, f"{prefix}_{name}.tif")
            res.rio.to_raster(out_file, compress='lzw', dtype=np.float32)
        
        return f"✅ Done: {prefix} (Valid: {count_da.max().values}/{max_years_count} yrs)"
        
    except Exception as e:
        return f"❌ Error on {tag}: {str(e)}"

# ส่วน MAIN LOGIC ใช้ตัวเดิมของนายได้เลยครับ แค่เปลี่ยนชื่อฟังก์ชันเรียกใช้

# ───── 3. MAIN LOGIC ──────────────────────────────────────────────

# สร้างรายการวันที่จะประมวลผล
doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

print("🏃 เริ่มประมวลผลแบบ Sequential (Simple Mode)...")

for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    current_out_dir = os.path.join(RETIRA_Majority_LC, f"{s}_{d}")
    
    for year, doy in doy_year_pairs:
        target_dt = find_file(DeltaT_LC, s, d, year, doy)
        target_lc = find_lc_file(outputLC_Folder, year)
        
        if target_dt and target_lc:
            tag = f"{year}_{doy:03d}"
            
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                base_years = range(year - (n_years - 1), year + 1)
                valid_dt_paths, valid_lc_paths = [], []
                
                for by in base_years:
                    dt_p = find_file(DeltaT_LC, s, d, by, doy)
                    lc_p = find_lc_file(outputLC_Folder, by)
                    if dt_p and lc_p:
                        valid_dt_paths.append(dt_p)
                        valid_lc_paths.append(lc_p)
                
                if len(valid_dt_paths) >= 2:
                    result = calculate_retira_with_debug(
                        valid_dt_paths, valid_lc_paths, target_dt, target_lc,
                        current_out_dir, tag, lbl, s, d
                    )
                    print(result)

print("🎉 === ประมวลผลเสร็จสิ้นทุกรายการ ===")

Continues

LC

In [ ]:
%%time
import os
import datetime
import numpy as np
import glob
import xarray as xr
import rioxarray
import warnings

# ปิด Warning เพื่อความสะอาดของหน้าจอ
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ───── 1. CONFIGURATION ──────────────────────────────────────────
scenarios = [
    {'sensor': 'MOD', 'DN': 'Day'},
    {'sensor': 'MOD', 'DN': 'Night'},
    {'sensor': 'MYD', 'DN': 'Day'},
    {'sensor': 'MYD', 'DN': 'Night'}
]

# ───── 2. FUNCTIONS ────────────────────────────────────────────────

def find_file(root, sensor, dn, year, doy):
    pattern = f"*{sensor}*{dn}*{year}*{doy:03d}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def find_lc_file(root, year):
    pattern = f"*LC*{year}*.tif"
    files = glob.glob(os.path.join(root, pattern))
    return files[0] if files else None

def calculate_retira_consecutive_debug(data_paths, lc_paths, out_dir, tag, lbl, sensor, dn):
    try:
        prefix = f"{sensor}_{dn}_{tag}_{lbl}Y"
        os.makedirs(out_dir, exist_ok=True)
        nodata_val = -9999.0
        
        # 1. โหลด Reference (ปีปัจจุบัน) เพื่อล็อคพิกัด (ไฟล์พันกว่าเมตร)
        # ในลิสต์ data_paths ตัวแรกคือปีปัจจุบัน (เพราะเราสั่ง reverse=True มาจากข้างนอก)
        ref_path = data_paths[0]
        ref_da = rioxarray.open_rasterio(ref_path, mask_and_scale=True).squeeze()
        if "band" in ref_da.dims: ref_da = ref_da.drop_vars("band")
        
        max_years_count = len(data_paths)

        # 2. โหลด LC Stack และตบพิกัด
        lc_list = []
        for p in lc_paths:
            da = rioxarray.open_rasterio(p).squeeze()
            da = da.rio.reproject_match(ref_da, resampling=0) # LC ต้อง Nearest
            lc_list.append(da)
        
        lc_stack = xr.concat(lc_list, dim="time")
        current_lc = lc_stack.isel(time=0) # ดึงปีปัจจุบันมาเป็นเกณฑ์
        
        # 3. ตรรกะความต่อเนื่อง (Consecutive Logic)
        # ถ้าพิกเซลไหน LC เปลี่ยนปุ๊บ cumprod จะทำให้ปีที่เหลือเป็น 0 (False) ทันที
        lc_match = (lc_stack == current_lc)
        valid_lc_mask = lc_match.astype(int).cumprod(dim="time").astype(bool)
        
        # 4. โหลด Data Stack และตบพิกัด
        data_list = []
        for p in data_paths:
            da = rioxarray.open_rasterio(p, mask_and_scale=True).squeeze()
            if da.rio.width != ref_da.rio.width or da.rio.height != ref_da.rio.height:
                da = da.rio.reproject_match(ref_da, resampling=1) # อุณหภูมิใช้ Bilinear
            data_list.append(da)
            
        data_stack = xr.concat(data_list, dim="time")
        
        # Apply Mask ความต่อเนื่อง
        masked_stack = data_stack.where(valid_lc_mask)
        
        # 5. คำนวณสถิติ
        # count_da = จำนวนปีที่ที่ดิน "ยังไม่เปลี่ยน" นับถอยหลังไป
        count_da = masked_stack.notnull().sum(dim="time")
        max_years_da = xr.full_like(ref_da, fill_value=max_years_count)
        
        mean_da = masked_stack.mean(dim="time")
        std_da = masked_stack.std(dim="time", ddof=1)
        
        current_da = data_stack.isel(time=0)

        # 6. คำนวณ RETIRA
        z_da = (current_da - mean_da) / std_da
        
        # เงื่อนไข NoData: ข้อมูลไม่พอ (< 2 ปี) หรือ STD เป็น 0
        final_mask = (count_da < 2) | (std_da == 0) | (current_da.isnull())
        
        # 7. เตรียมและบันทึก Output (เน้นแก้เรื่อง Count ไม่เซฟ)
        outputs = {
            "RETIRA": z_da.where(~final_mask, nodata_val),
            "Count": count_da.astype(np.float32),      # จำนวนปีที่ต่อเนื่องจริง
            "MaxYears": max_years_da.astype(np.float32) # จำนวนปีที่พยายามจะเอามาใช้
        }

        for name, da in outputs.items():
            res = da.load() # บังคับคำนวณเข้า RAM
            res.rio.write_nodata(nodata_val, encoded=True, inplace=True)
            res.rio.set_crs(ref_da.rio.crs, inplace=True)
            
            out_file = os.path.join(out_dir, f"{prefix}_{name}.tif")
            res.rio.to_raster(out_file, compress='lzw', dtype=np.float32)
            
        return f"✅ Done (Consecutive): {prefix} (Continuous: {count_da.max().values}/{max_years_count})"
    
    except Exception as e:
        return f"❌ Error @ {tag}: {str(e)}"

# ───── 3. MAIN LOOP ───────────────────────────────────────────────

doy_year_pairs = []
curr = date_start
while curr <= date_end:
    doy_year_pairs.append((curr.year, curr.timetuple().tm_yday))
    curr += datetime.timedelta(days=1)

print("🚀 เริ่มประมวลผล Consecutive LC แบบล็อคพิกัดต้นฉบับ...")

for sc in scenarios:
    s, d = sc['sensor'], sc['DN']
    current_out_dir = os.path.join(RETIRA_Continue_LC, f"{s}_{d}")
    
    for year, doy in doy_year_pairs:
        target_file = find_file(DeltaT_LC, s, d, year, doy)
        
        if target_file:
            tag = f"{year}_{doy:03d}"
            
            for lbl, n_years in {"3":3, "5":5, "10":10}.items():
                # เรียงปีจาก ปัจจุบัน -> อดีต (สำคัญมากสำหรับ Consecutive Logic)
                base_years = sorted(range(year - (n_years - 1), year + 1), reverse=True)
                
                valid_data_paths = []
                valid_lc_paths = []
                
                for by in base_years:
                    p_data = find_file(DeltaT_LC, s, d, by, doy)
                    p_lc = find_lc_file(outputLC_Folder, by)
                    
                    if p_data and p_lc:
                        valid_data_paths.append(p_data)
                        valid_lc_paths.append(p_lc)
                    else:
                        break # ถ้าปีถัดไปไม่มีข้อมูล ให้ตัดสายทันที
                
                if len(valid_data_paths) >= 2:
                    result = calculate_retira_consecutive_debug(
                        valid_data_paths, valid_lc_paths, 
                        current_out_dir, tag, lbl, s, d
                    )
                    print(result)

print("🎉 จบงาน! ลองเช็คไฟล์ _Count เทียบกับ _MaxYears ดูนะนาย")

Cutoff Value with Interesting

In [ ]:
# import os
# import rioxarray
# import xarray as xr

# # --- CONFIG ---
# input_dir = RETIRA_Clip 
# output_dir = RETIRA_Final
# threshold_limit = 0
# nodata_value = -9999

# # --- PROCESS ---
# for root, dirs, files in os.walk(input_dir):
#     for file in files:
#         # เช็คเฉพาะไฟล์ที่ลงท้ายด้วย _RETIRA.tif เท่านั้น
#         if file.lower().endswith('_retira.tif'):
            
#             input_path = os.path.join(root, file)
            
#             # จัดการ Path ขาออกให้ล้อตามโครงสร้าง Folder เดิม
#             rel_path = os.path.relpath(root, input_dir)
#             target_folder = os.path.join(output_dir, rel_path)
#             os.makedirs(target_folder, exist_ok=True)
#             output_path = os.path.join(target_folder, file)

#             try:
#                 print(f"⌛ Processing: {file}")
                
#                 # 1. เปิดไฟล์
#                 rds = rioxarray.open_rasterio(input_path, chunks=True)
                
#                 # 2. คัดค่าเฉพาะที่ < threshold และเปลี่ยนค่าอื่น (รวมถึง NaN เดิม) ให้เป็น NaN ใหม่ทั้งหมด
#                 # ขั้นตอนนี้จะทำให้พวกค่าขยะหรือค่าที่น้อยกว่าเกณฑ์หายไป
#                 rds_filtered = rds.where(rds < threshold_limit)
                
#                 # 3. เขียน Header ลงไปใน Metadata ว่า NoData คือ -9999
#                 # เพื่อให้ ArcGIS Pro อ่านแล้วรู้ว่าพิกเซลว่างๆ คือความโปร่งใส
#                 rds_filtered.rio.write_nodata(nodata_value, inplace=True)
                
#                 # 4. Save ไฟล์พร้อมย้ำ NoData ใน Header ของ GeoTIFF อีกรอบ
#                 rds_filtered.rio.to_raster(
#                     output_path,
#                     nodata=nodata_value,
#                     tiled=True,
#                     compress="lzw"
#                 )
                
#                 print(f"✅ Success: {output_path}")
                
#                 rds.close()
                
#             except Exception as e:
#                 print(f"❌ Error at {file}: {e}")

# print("\n🏁 เสร็จหมดทุกไฟล์แล้ว ไปเปิดใน ArcGIS ได้เลย!")